In [ ]:
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service as ChromeService
from selenium.webdriver.support.ui import Select
from webdriver_manager.chrome import ChromeDriverManager

driver = webdriver.Chrome(service=ChromeService(ChromeDriverManager().install()))
driver.implicitly_wait(10)
driver.get('http://localhost:3000')

driver.find_element(By.XPATH, "//button[contains(.,'Register') or contains(.,'NID') or contains(.,'VERIFY NID')]").click()
time.sleep(1)

uid = str(int(time.time()))[-5:]
driver.find_element(By.XPATH, "//form//input[@type='text']").send_keys(f"199226920150{uid}")

try:
    Select(driver.find_element(By.XPATH, "//select[contains(@aria-label,'day') or contains(@aria-label,'Birth day')]")).select_by_value("15")
    Select(driver.find_element(By.XPATH, "//select[contains(@aria-label,'month') or contains(@aria-label,'Birth month')]")).select_by_value("05")
    driver.find_element(By.XPATH, "//input[contains(@aria-label,'year') or contains(@placeholder,'Year')]").send_keys("1992")
except Exception as e:
    print("DOB select notice:", e)

driver.find_element(By.XPATH, "//button[contains(.,'Verify National Identity')]").click()
time.sleep(2)

try:
    driver.find_element(By.XPATH, "//button[contains(.,'Confirm') or contains(.,'Create Credentials')]").click()
    time.sleep(1)
except Exception:
    pass

try:
    phone_input = driver.find_element(By.XPATH, "//form//input[1]")
    phone_input.clear()
    phone_input.send_keys(f"01711{uid}")

    driver.find_element(By.XPATH, "//button[contains(.,'Verify Email')]").click()
    time.sleep(1.5)

    try:
        driver.find_element(By.XPATH, "//button[contains(.,'Auto-Fill')]").click()
        time.sleep(0.5)
    except Exception:
        otp_input = driver.find_element(By.XPATH, "//input[contains(@placeholder,'482910') or @maxLength='6']")
        otp_input.send_keys("123456")

    driver.find_element(By.XPATH, "//button[contains(.,'Confirm Code')]").click()
    time.sleep(1)

    pass_fields = driver.find_elements(By.XPATH, "//form//input[@type='password']")
    if pass_fields:
        pass_fields[0].send_keys("demo1234")
        if len(pass_fields) > 1:
            pass_fields[1].send_keys("demo1234")

    driver.find_element(By.XPATH, "//form//button[@type='submit' or contains(.,'Complete Registration')]").click()
    time.sleep(3)
except Exception as e:
    print("Step 3 notice:", e)

body_text = driver.find_element(By.TAG_NAME, 'body').text.lower()
assert any(kw in body_text for kw in ["success", "dashboard", "citizen", "welcome", "registered", "verified", "active"]), "registration failed"
print('[OK] Citizen Registration test passed')
driver.quit()